In [1]:
import importlib.util
from pathlib import Path

def _import_uatr_path():
    seeds = []
    p = globals().get("__vsc_ipynb_file__")
    if isinstance(p, str):
        seeds.append(Path(p).resolve().parent)
    cwd = Path.cwd().resolve()
    seeds.extend([cwd, *cwd.parents])
    for base in seeds:
        for cand in (
            base / "_uatr_path.py",
            base / "dev" / "research_nbs" / "_uatr_path.py",
        ):
            if cand.is_file():
                return cand
    raise RuntimeError(
        "Cannot find _uatr_path.py. Save/pull the repo and open the UATR folder in the editor."
    )

_spec = importlib.util.spec_from_file_location("_uatr_path", _import_uatr_path())
_up = importlib.util.module_from_spec(_spec)
assert _spec.loader is not None
_spec.loader.exec_module(_up)
_up.ensure_repo_on_sys_path()

import dev.utils as ut

In [2]:
# imports and settings

import os
import time
import pickle
import warnings
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from copy import deepcopy

import numpy as np
from numpy import linalg as LA
from numpy import histogram2d

from scipy import signal
from scipy.fft import fft, fftfreq, fftshift
from scipy.signal import find_peaks, butter, filtfilt, welch
from scipy.ndimage import gaussian_filter
from scipy.io import wavfile
from scipy.stats import wasserstein_distance_nd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

%load_ext autoreload
%autoreload 2

# do not show warnings
warnings.filterwarnings("ignore")

print("Imports complete.")

Imports complete.


In [11]:
path = _up.repo_root() / "data" / "ds2" / "pos" / "dpv1_1m.wav"
fs, data = wavfile.read(str(path))

In [16]:
# parameters and processing loop
height = 800
width = 1400
font_size = 16
fft_nperseg = 16000
percent_overlap = 0.25
window = 'hamming'
remove_dc = 20
crop_freq = 4000  # Hz, None to disable
normalization_window_size = 9  # in samples, None to disable
Q = 5

f_list = [636, 588, 900, 2000]

In [ ]:
F, T, Sxx, phase = ut.calc_spectrogram(data, fs, nperseg=fft_nperseg, percent_overlap=percent_overlap, window=window, remove_dc=remove_dc)

fig = make_subplots(rows=4, cols=len(f_list), column_titles=[f"f={f} Hz" for f in f_list], row_titles=["Phase Dist.", "Trans. Size Dist.", "Trans. Dist.", "S2G Graph"])
for ix, f in enumerate(f_list, start=1):
    f_ix = np.argmin(np.abs(F - f))
    f_phase = phase[f_ix, :]

    # distribution of phases 
    hist, bin_edges = np.histogram(f_phase, bins=Q, range=(-np.pi, np.pi), density=True)
    fig.add_trace(go.Bar(x=bin_edges[:-1], y=hist, name='Phase Distribution', marker_color='blue', showlegend=False), row=1, col=ix)

    # distribution of delta phases
    delta_phase = np.diff(f_phase)
    delta_phase = np.abs(delta_phase)
    hist_delta, bin_edges_delta = np.histogram(delta_phase, bins=Q, range=(0, 2 * np.pi), density=True)
    fig.add_trace(go.Bar(x=bin_edges_delta[:-1], y=hist_delta, name='Delta Phase Distribution', marker_color='orange', showlegend=False), row=2, col=ix)

    # distribution of traqnsitions
    x_transitions = ut.get_s2g(f_phase, n_levels=Q)
    fig.add_trace(go.Heatmap(z=x_transitions, colorscale='Viridis', showscale=False), row=3, col=ix)

    # graph from S2G
    G = ut.get_s2g_graph(f_phase, n_levels=Q)
    fig_g = ut.draw_graph(G)
    for tr in fig_g.data:
                fig.add_trace(deepcopy(tr), row=4, col=ix)

fig.update_layout(height=800, width=1400, title_text='Phase Distribution, Phase Transition-Size Distribution, Phase Transition Distribution, and S2G Graph for Selected Frequencies', font=dict(size=12))
fig.write_image("/Users/itamarmerfeld/Projects/UATR/results/phase_distribution_analysis.pdf")
fig.show()

In [42]:
F, T, Sxx, phase = ut.calc_spectrogram(data, fs, nperseg=fft_nperseg, percent_overlap=percent_overlap, window=window, remove_dc=remove_dc, crop_freq=crop_freq)
Q = 100

fig = make_subplots(rows=3, 
                    cols=len(f_list), 
                    shared_yaxes=True, 
                    vertical_spacing=0.05,
                    column_titles=[f"f={f} Hz" for f in f_list], 
                    row_titles=["Phase Dist.", "Trans. Size Dist.", "Trans. Dist.", "S2G Graph"])
for ix, f in enumerate(f_list, start=1):
    f_ix = np.argmin(np.abs(F - f))
    f_phase = phase[f_ix, :]

    # distribution of phases 
    hist, bin_edges = np.histogram(f_phase, bins=Q, range=(-np.pi, np.pi), density=True)
    # fig.add_trace(go.Bar(x=bin_edges[:-1], y=hist, name='Phase Distribution', marker_color='blue', showlegend=False), row=1, col=ix)
    fig.add_trace(go.Scatter(x=bin_edges[:-1], y=hist, name='Phase Distribution', marker_color='blue', showlegend=False), row=1, col=ix)

    # distribution of delta phases
    delta_phase = np.diff(f_phase)
    delta_phase = np.abs(delta_phase)
    hist_delta, bin_edges_delta = np.histogram(delta_phase, bins=Q, range=(0, 2 * np.pi), density=True)
    # fig.add_trace(go.Bar(x=bin_edges_delta[:-1], y=hist_delta, name='Delta Phase Distribution', marker_color='orange', showlegend=False), row=2, col=ix)
    fig.add_trace(go.Scatter(x=bin_edges_delta[:-1], y=hist_delta, name='Delta Phase Distribution', marker_color='orange', showlegend=False), row=2, col=ix)

    # distribution of traqnsitions
    x_transitions = ut.get_s2g(f_phase, n_levels=Q)
    fig.add_trace(go.Heatmap(z=x_transitions, colorscale='Viridis', showscale=False), row=3, col=ix)

fig.update_layout(height=800, width=1000, font=dict(size=12))
fig.write_image("/Users/itamarmerfeld/Projects/UATR/results/phase_distribution_analysis_2.png", scale=10)
fig.show()